In [0]:
-- CTE filters early before the join, reducing shuffle size
WITH active_shows AS (
    -- Predicate pushed down to silver_shows scan — eliminates nulls before join
    SELECT show_id, show_name, genre
    FROM de_assessment_dev.silver.silver_shows
    WHERE genre IS NOT NULL AND genre != 'Unknown'
),
episode_stats AS (
    -- Filter runtime early — avoids carrying nulls through aggregation
    SELECT show_id, season, COUNT(*) AS ep_count, AVG(runtime) AS avg_rt
    FROM de_assessment_dev.silver.silver_episodes
    WHERE runtime > 0
    GROUP BY show_id, season
)
SELECT
    a.show_name,
    a.genre,
    e.season,
    e.ep_count,
    ROUND(e.avg_rt, 1) AS avg_runtime_mins
FROM active_shows a
JOIN episode_stats e ON a.show_id = e.show_id
ORDER BY a.genre, e.ep_count DESC;